In [46]:
import sqlite3
conn = sqlite3.connect('hospital.db')
cursor = conn.cursor()

# Get all table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

# Delete all data from each table
for table in tables:
    cursor.execute(f"DELETE FROM {table[0]}")
    print(f"Cleared: {table[0]}")

conn.commit()
conn.close()

print("Done!")

Cleared: patients
Done!


In [47]:
import pandas as pd

def fill_missing_ward(df):
    # Create mapping: most common ward for each disease (excluding 'Under Diagnosis')
    ward_map = df[df['Disease'] != 'Under Diagnosis'] \
        .groupby('Disease')['Ward'] \
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unassigned')
    # Fill missing Ward using mapping
    df['Ward'] = df.apply(
        lambda row: ward_map.get(row['Disease'], 'Unassigned')
        if pd.isna(row['Ward']) and row['Disease'] != 'Under Diagnosis'
        else row['Ward'],
        axis=1
    )
    # Fill remaining missing with Unassigned
    df['Ward'] = df['Ward'].fillna('Unassigned')
    return df

def convert_dates(df):
    date_cols = ['AdmissionDate', 'DischargeDate', 'DateOfBirth']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    df['LengthOfStay'] = (df['DischargeDate'] - df['AdmissionDate']).dt.days
    df['LengthOfStay'] = df['LengthOfStay'].fillna(0)
    for col in date_cols:
        df[col] = df[col].dt.strftime('%Y-%m-%d').fillna('Not Available')
    return df

def fill_missing_room(df):
    # Find most common room for each ward
    room_map = df.groupby('Ward')['RoomNumber'].agg(
        lambda x: x.mode()[0] if not x.mode().empty else 'Not Assigned'
    )
    # Fill missing RoomNumber using ward's most common room
    df['RoomNumber'] = df.apply(
        lambda row: room_map.get(row['Ward'], 'Not Assigned')
        if pd.isna(row['RoomNumber']) else row['RoomNumber'],
        axis=1
    )
    # If still missing, fill with Not Assigned
    df['RoomNumber'] = df['RoomNumber'].fillna('Not Assigned')
    return df

In [48]:
# CELL 2 - ETL Process + SQLite Storage (hospital.db) + Logging
import pandas as pd
import sqlite3
import logging

#  Correct Logging Setup
logger = logging.getLogger('hospital_etl')
logger.setLevel(logging.INFO)

if not logger.handlers:
    file_handler = logging.FileHandler('hospital_etl.log')
    file_handler.setLevel(logging.INFO)
    file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(file_handler)

def etl_process():
    try:
        logger.info("ETL process started")
        # EXTRACT
        df = pd.read_csv("hospital_patient_dataset.csv")
        logger.info(f"Extracted {len(df)} records from hospital_patient_dataset.csv")
        print(f"Extracted {len(df)} records")

        # TRANSFORM
        df['AdmissionDate'] = pd.to_datetime(df['AdmissionDate'], errors='coerce')
        df['DischargeDate']  = pd.to_datetime(df['DischargeDate'],  errors='coerce')
        df['DateOfBirth']    = pd.to_datetime(df['DateOfBirth'],    errors='coerce')

        df['FirstName']     = df['FirstName'].fillna('Unknown')
        df['LastName']      = df['LastName'].fillna('Unknown')
        df['Gender']        = df['Gender'].fillna('Unknown')
        df['ContactNumber'] = df['ContactNumber'].fillna('Not Available')
        df['Email']         = df['Email'].fillna('Not Available')
        df['Address']       = df['Address'].fillna('Unknown')
        df['Disease']       = df['Disease'].fillna('Under Diagnosis')

        df = fill_missing_ward(df)
        df = fill_missing_room(df)
        df = convert_dates(df)

        df['Gender'] = df['Gender'].replace({'M': 'Male', 'F': 'Female'})
        df['StayStatus'] = df['LengthOfStay'].apply(
            lambda x: 'Discharged' if x > 0 else 'Admitted / Ongoing'
        )

        logger.info("Transformation completed successfully")
        print("Transformation completed")
        df.to_csv("hospital_patient_cleaned_dataset.csv")

        conn = sqlite3.connect('hospital.db')
        df.to_sql('patients', conn, if_exists='replace', index=False)
        conn.close()

        logger.info("Data loaded into hospital.db (table: patients)")
        print("\nData loaded into hospital.db successfully")

    except Exception as e:
        logger.error(f"ETL failed: {e}")
        print(f"ETL failed: {e}")

In [49]:
# CELL 3 - Scheduling
import schedule
import time

def run_etl_job():
    print("Scheduler: Running ETL job...")
    logger.info("Scheduled ETL job triggered")   #  logging → logger
    etl_process()
    logger.info("Scheduled ETL job completed")   #  logging → logger
    print("Scheduler: ETL job done.")

# Schedule every day at 02:00 (Production)
# schedule.every().day.at("02:00").do(run_etl_job)

# Test: every 1 minute
# schedule.every(1).minutes.do(run_etl_job)
schedule.every(5).seconds.do(run_etl_job)

# print("Scheduler started... ETL runs every 1 minute")
print("Scheduler started... ETL runs every 5 second")

# while True:
#     schedule.run_pending()
#     time.sleep(10) #10 second
try:
    for _ in range(3):  # only 3 times run 
        schedule.run_pending()
        time.sleep(10)
except KeyboardInterrupt:
    print("Scheduler stopped by user")


Scheduler started... ETL runs every 5 second
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded into hospital.db successfully
Scheduler: ETL job done.
Scheduler: Running ETL job...
Extracted 20 records
Transformation completed

Data loaded in

In [50]:
# CELL 4 - Logging & Monitoring Check
import os

log_file = 'hospital_etl.log'

if os.path.exists(log_file):
    print("=== PIPELINE LOG ===")
    with open(log_file, 'r') as f:
        print(f.read())
else:
    print("No log file found yet. Run the ETL cell first.")

=== PIPELINE LOG ===
2026-03-29 22:58:09,094 - INFO - Scheduled ETL job triggered
2026-03-29 22:58:09,095 - INFO - ETL process started
2026-03-29 22:58:09,100 - INFO - Extracted 20 records from hospital_patient_dataset.csv
2026-03-29 22:58:09,129 - INFO - Transformation completed successfully
2026-03-29 22:58:09,151 - INFO - Data loaded into hospital.db (table: patients)
2026-03-29 22:58:09,152 - INFO - Scheduled ETL job completed
2026-03-29 22:58:09,152 - INFO - Scheduled ETL job triggered
2026-03-29 22:58:09,152 - INFO - ETL process started
2026-03-29 22:58:09,156 - INFO - Extracted 20 records from hospital_patient_dataset.csv
2026-03-29 22:58:09,185 - INFO - Transformation completed successfully
2026-03-29 22:58:09,207 - INFO - Data loaded into hospital.db (table: patients)
2026-03-29 22:58:09,208 - INFO - Scheduled ETL job completed
2026-03-29 22:58:09,208 - INFO - Scheduled ETL job triggered
2026-03-29 22:58:09,208 - INFO - ETL process started
2026-03-29 22:58:09,213 - INFO - Extr

In [51]:
# CELL 5 - Report Generation

import sqlite3
import pandas as pd

conn = sqlite3.connect('hospital.db')
df = pd.read_sql_query("SELECT * FROM patients", conn)

# REPORT 1: Patients per Ward
ward_report = df.groupby('Ward').agg(
    TotalPatients=('PatientID', 'count'),
    AvgStay=('LengthOfStay', 'mean')
).round(1)

ward_report.to_csv('ward_report.csv')
print("=== REPORT 1: Patients per Ward ===")
print(ward_report)

# REPORT 2: Disease Summary
disease_report = df.groupby('Disease').agg(
    TotalPatients=('PatientID', 'count'),
    AvgStay=('LengthOfStay', 'mean')
).round(1).sort_values('TotalPatients', ascending=False)

disease_report.to_csv('disease_report.csv')
print("\n=== REPORT 2: Disease Summary ===")
print(disease_report)

# REPORT 3: Gender Distribution

gender_report = df['Gender'].value_counts()
gender_report.to_csv('gender_report.csv')
print("\n=== REPORT 3: Gender Distribution ===")
print(gender_report)

# REPORT 4: Missing Data Report
missing_report = df.isnull().sum()
missing_report.to_csv('missing_report.csv')
print("\n=== REPORT 4: Missing Data Report ===")
print(missing_report)

conn.close()
print("\nAll reports generated successfully!")

=== REPORT 1: Patients per Ward ===
      TotalPatients  AvgStay
Ward                        
A                 6      2.0
B                 7      2.3
C                 4      4.5
D                 3      2.3

=== REPORT 2: Disease Summary ===
                        TotalPatients  AvgStay
Disease                                       
Asthma                              2      3.5
Dengue                              2      2.0
Diarrhea                            2      1.5
Diabetes                            2      0.0
Pneumonia                           2      5.5
Typhoid                             2      4.0
Chronic Kidney Disease              1      0.0
Anemia                              1      2.0
Food Poisoning                      1      0.0
Gastritis                           1      2.0
Hypertension                        1      2.0
Heart Disease                       1      7.0
Malaria                             1      3.0
Under Diagnosis                     1      4.0

==